In [146]:
sc.stop()
from pyspark import SparkContext
import findspark
findspark.init()

import numpy as np
import warnings
warnings.filterwarnings("ignore")

sc = SparkContext(master="local[*]", appName="TextFileExample")

DATA_FILE = "botnet_sample_1000.csv"
X_SIZE = 11
Y_SIZE = 1
N_ITER = 10
LEARNING_RATE = 1.5


25/10/11 12:01:59 ERROR Executor: Exception in task 0.0 in stage 22.0 (TID 42): Connection reset
25/10/11 12:01:59 ERROR Executor: Exception in task 1.0 in stage 22.0 (TID 43): Connection reset


In [236]:
# def readFile (filename):
# Arguments:
# filename – name of the spam dataset file
# 12 columns: 11 features/dimensions (X) + 1 column with labels (Y)
# Y -- Train labels (0 if normal traffic, 1 if botnet)
# m rows: number of examples (m)
# Returns:
# An RDD containing the data of filename. Each example (row) of the file
# corresponds to one RDD record. Each record of the RDD is a tuple (X,y).
# “X” is an array containing the 11 features (float number) of an example
# “y” is the 12th column of an example (integer 0/1)

def readFile(filename):
    rdd = sc.textFile(filename)
    def map_line(line):
        elements = [float(element) for element in line.split(",")]
        return (np.array(elements[:-1]), int(elements[-1]))
    return rdd.map(map_line)



# def normalize (RDD_Xy):
# Arguments:
# RDD_Xy is an RDD containing data examples. Each record of the RDD is a tuple
# (X,y).
# “X” is an array containing the 11 features (float number) of an example
# “y” is the label of the example (integer 0/1)
# Returns:
# An RDD rescaled to N(0,1) in each column (mean=0, standard deviation=1)
def normalize (RDD_Xy):
    
    rdd_col = RDD_Xy.map(lambda xy: (np.array(xy[:-1], dtype=float).flatten(), int(xy[-1])))


    sum_vec = rdd_col.map(lambda xy: xy[0]).reduce(lambda a, b: a + b)
    media = sum_vec / n
    
    
    varianza = rdd_col.map(lambda v: (v[0]-media)**2).reduce(lambda a,b:a+b)/n
    
    std=np.sqrt(varianza)
    #Normalize
    
    norm = rdd_col.map(lambda v: ((v[0] - media)/std, v[1]))
    
    return norm


# def train (RDD_Xy, iterations, learning_rate, lambda_reg):
# Arguments:
# RDD_Xy --- RDD containing data examples. Each record of the RDD is a tuple
# (X,y).
# “X” is an array containing the 11 features (float number) of an example
# “y” is the label of the example (integer 0/1)
# iterations -- number of iterations of the optimization loop
# learning_rate -- learning rate of the gradient descent
# lambda_reg – regularization rate
# Returns:
# A list or array containing the weights “w” and bias “b” at the end of the
# training process


def train(RDD_Xy, iterations, learning_rate):
    
    sigma = lambda z : (1 / (1 + (np.e**(-z))))
    
    W = np.zeros(X_SIZE)
    b = 0
    
    def calculate_dw(rdd, W, b):
        rdd = rdd.map(lambda xy: np.array([(sigma(np.dot(W, xy[0]) + b) - xy[1]) * x_i for x_i in xy[0]]))
        dw = rdd.reduce(lambda a, b : a + b) / n
        return dw
    
    def calculate_db(rdd, W, b):
        rdd = rdd.map(lambda xy: sigma(np.dot(W, xy[0]) + b) - xy[1])
        db = rdd.reduce(lambda a, b : a + b) / n
        return db
    
    for _ in range(iterations):
        dw = calculate_dw(RDD_Xy, W, b)
        db = calculate_db(RDD_Xy, W, b)
        W = W - learning_rate * dw
        b = b - learning_rate * db
    
    return W, b


# def accuracy (w, b, RDD_Xy):
# Arguments:
# w -- weights
# b -- bias
# RDD_Xy – RDD containing examples to be predicted
# Returns:
# accuracy -- the number of predictions that are correct divided by the number
# of records (examples) in RDD_xy.
# Predict function can be used for predicting a single example
def accuracy(w, b, RDD_Xy):
    predictions = RDD_Xy.map(lambda v: predict(w, b, v[0]))
    count = predictions.reduce(lambda a, b : a + b)
    return count/n


# def predict (w, b, X):
# Arguments:
# w -- weights
# b -- bias    correct = RDD_Xy.map(lambda v: predict(w, b, v[0]))
# X – Example to be predicted
# Returns:
# Y_pred – a value (0/1) corresponding to the prediction of X

def predict(w, b, data):
    sigma = lambda z : (1 / (1 + (np.e**(-z))))
    res = sigma((w*data[0]).sum() + b)
    if(res>=0.5): 
        return 1
    else:
        return 0



In [237]:
# read data
data = readFile(DATA_FILE)
n = data.count()

# standarize
rdd_norm=normalize(data)

W, b = train(rdd_norm, N_ITER, LEARNING_RATE)


acc = accuracy(W,b, rdd_norm)
print(acc)
#print("Accuracy: ", acc)

0.64
